# Evaluación de video RT-DETR Ecuador

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="ecuador"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Cruce contra referencia por intervalos
Requiere `plate_gt,frame_start,frame_end` y predicciones `frame_index,plate_pred`. Solo se comparan predicciones dentro de intervalos anotados; no mide falsos negativos fuera de los intervalos.

In [ ]:
import pandas as pd
from src.data_processing import normalize_plate
PRED="/content/decisions_rtdetr_ecuador.csv"
GT=REPO_ROOT/"evaluation/ecuador/blind_evaluation/video_ground_truth.csv"
pred=pd.read_csv(PRED); gt=pd.read_csv(GT)
need={"plate_gt","frame_start","frame_end"}
if not need.issubset(gt.columns): raise ValueError(f"Faltan columnas: {need-set(gt.columns)}")
def ref(frame):
    rows=gt[(gt.frame_start<=frame)&(gt.frame_end>=frame)]
    return rows.iloc[0].plate_gt if len(rows)==1 else None
pred["plate_gt"]=pred.frame_index.map(ref); scored=pred[pred.plate_gt.notna()].copy()
scored["exact_match"]=scored.apply(lambda r:normalize_plate(r.plate_gt)==normalize_plate(r.plate_pred),axis=1)
print({"n_predictions":len(pred),"n_scored":len(scored),"n_unmatched":len(pred)-len(scored),
       "exact_match_rate":float(scored.exact_match.mean()) if len(scored) else None})
scored.to_csv("/content/evaluacion_video_rtdetr_ecuador.csv",index=False)
